- import libraries: torch, torchvision
- load mnist dataset: transform it, convert it to batches
- set up nn model using nn.Module
- set up training loop and evaluation after every 1000 epoch

In [1]:
## Import necessary libraries
import torch
import torchvision

from torch import nn
from torch import optim
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST

In [2]:
## make default device as cuda
device = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device)
print(device)

cpu


In [3]:
## Download data and apply transform
transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.0],std=[1.0]),
    ]
)

training_data = MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

testing_data = MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 43.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.42MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.1MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.1MB/s]


In [4]:
print(training_data[0][0].shape) ## <<-- image
print(training_data[0][1]) ## <<-- label

torch.Size([1, 28, 28])
5


In [5]:
## Convert dataset into batches using dataloader
training_data_loader = DataLoader(
    dataset=training_data,
    batch_size=32,
    shuffle=True,
)

testing_data_loader = DataLoader(
    dataset=testing_data,
    batch_size=32,
    shuffle=False,
)

In [6]:
## Model
class MnistClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 500)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(500, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x


model = MnistClassificationModel().to(device)

In [7]:
## Loss Function and Optimizer
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    params=model.parameters(),
)

In [9]:
# ## Setup - training loop
# for epoch in range(3):
#     model.train()
#     train_loss_sum = 0
#     total_train_points = 0

#     for train_batch_index, (train_input, train_target) in enumerate(training_data_loader):
#         optimizer.zero_grad()

#         train_input = train_input.to(device)
#         train_target = train_target.to(device)

#         train_output = model(train_input)
#         train_loss = loss_function(train_output, train_target)

#         batch_size = train_target.size(0)
#         total_train_points += batch_size
#         train_loss_sum += (train_loss.item() * batch_size)

#         train_loss.backward()
#         optimizer.step()

#     if (epoch)%1 == 0:
#         train_loss_avg = train_loss_sum/total_train_points
#         print(f"epoch = {epoch} | train_loss_avg = {train_loss_avg}")

#     if (epoch+1)%1 == 0:
#         model.eval()
#         test_loss_sum = 0
#         total_test_points = 0

#         with torch.no_grad():
#             for test_batch_index,(test_input, test_target) in enumerate(testing_data_loader):

#                 test_input = test_input.to(device)
#                 test_target = test_target.to(device)

#                 test_output = model(test_input)
#                 test_loss = loss_function(test_output, test_target)

#                 batch_size = test_target.size(0)
#                 total_test_points += batch_size
#                 test_loss_sum += (test_loss.item() * batch_size)

#             test_loss_avg = test_loss_sum/total_test_points
#             print(f"<--->\nepoch = {epoch} | test_loss_avg = {test_loss_avg}\n<--->")


In [48]:
def model_train(model, training_data_loader, optimizer, device, epoch):
    model.train()
    train_loss_sum = 0
    total_train_points = 0

    for train_batch_index, (train_input, train_target) in enumerate(training_data_loader):
        optimizer.zero_grad()

        train_input = train_input.to(device)
        train_target = train_target.to(device)

        train_output = model(train_input)
        train_loss = loss_function(train_output, train_target)

        batch_size = train_target.size(0)
        total_train_points += batch_size
        train_loss_sum += (train_loss.item() * batch_size)

        train_loss.backward()
        optimizer.step()

    if (epoch)%1 == 0:
        train_loss_avg = train_loss_sum/total_train_points
        print(f"epoch = {epoch} | train_loss_avg = {train_loss_avg}")


def model_test(model, testing_data_loader, device, epoch):
    if (epoch)%1 == 0:
        model.eval()
        test_loss_sum = 0
        total_test_points = 0
        test_loss_avg = 0
        correctly_predicted_points = 0
        accuracy = 0

        with torch.no_grad():
            for test_batch_index,(test_input, test_target) in enumerate(testing_data_loader):

                test_input = test_input.to(device)
                test_target = test_target.to(device)

                test_output = model(test_input)


                ### Debugging done to find out how to count number of correct predictions: :)
                #### Debug start vvvv
                # # print(f"test_output = \n---\n{test_output}\n---")
                # print(f"test_output.size() = {test_output.size()}")
                # print(f"test_target.size() = {test_target.size()}")

                # print("\n---")
                # print(f"torch.argmax(input=test_output, dim=1) = {torch.argmax(input=test_output, dim=1)}")
                # print(f"torch.argmax(input=test_output, dim=1) = {torch.argmax(input=test_output, dim=1).size()}")
                # print("---")

                # print("\n---")
                # print(f"test_target = {test_target}")
                # print(f"test_target.size() = {test_target.size()}")
                # print("---")

                # print(torch.argmax(input=test_output, dim=1)==test_target)
                #### Debug stop ^^^^

                predicted_label = torch.argmax(input=test_output, dim=1)
                comparison_tensor = predicted_label==test_target
                correctly_predicted_points += torch.sum(comparison_tensor).item()
                # print(f"correctly_predicted_points = {correctly_predicted_points}")

                test_loss = loss_function(test_output, test_target)

                batch_size = test_target.size(0)
                total_test_points += batch_size
                test_loss_sum += (test_loss.item() * batch_size)

            test_loss_avg = test_loss_sum/total_test_points
            accuracy = correctly_predicted_points/total_test_points
            print(f"<--->\nepoch = {epoch} | test_loss_avg = {test_loss_avg} | accuracy = {accuracy}\n<--->")

In [52]:
## Setup - training loop
for epoch in range(3):
    model_train(model, training_data_loader, optimizer, device, epoch)
    model_test(model, testing_data_loader, device, epoch)


epoch = 0 | train_loss_avg = 0.012571738087080061
<--->
epoch = 0 | test_loss_avg = 0.0849874975918805 | accuracy = 0.9802
<--->
epoch = 1 | train_loss_avg = 0.010851399474461808
<--->
epoch = 1 | test_loss_avg = 0.08668789790405608 | accuracy = 0.9797
<--->
epoch = 2 | train_loss_avg = 0.01028560595814876
<--->
epoch = 2 | test_loss_avg = 0.08151814961869698 | accuracy = 0.9816
<--->


In [53]:
# Save the learned weights
torch.save(model.state_dict(), "model.pth")

In [54]:
# Save the checkpoint: epoch, model state, optimizer state

checkpoint = {
    "epoch": epoch,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
}

torch.save(checkpoint, f"checkpoint_epoch_{epoch}.pth")

In [51]:
## To load the model weights and optimizer state from save dict
checkpoint = None
checkpoint = torch.load("checkpoint_epoch_2.pth")

model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])